## Segmentation Pipeline ##
**Project**: Artificial Intelligence-Based Cell Survival Colony Counting \
**Author**: Mona Wang \
**Supervisors**: Laya Rafiee Sevyeri, Shirin A. Enger 

This Jupyter notebook will:
1. Segment the colonies based on watershed algorithm
1. Output the final colony count
1. Output the image of the segmented colony

In [1]:
## Toggles ##
segment_colonies = True
count_colonies = True
test_accuracy = True

core = False
solidity = False

In [2]:
## Imports ##
import numpy as np
import glob
import re

# Image Processing
import cv2 as cv
from skimage import morphology, img_as_ubyte
from skimage.morphology import extrema
from skimage.util import img_as_float64

# For controlling cell execution
from IPython.core.magic import register_cell_magic
from IPython import get_ipython

@register_cell_magic
def skip_if(line, cell):
    if eval(line):
        return
    get_ipython().run_cell(cell)

## Fix the issue of Error #15 "Initializing libiomp5md.dll, but found mk2iomp5md.dll already initialized." ##
import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"


## Function for Numerical Ordering ##
numbers = re.compile(r'(\d+)')
def numericalSort(value):
    parts = numbers.split(value)
    parts[1::2] = map(int, parts[1::2])
    return parts

In [7]:
## Paths ##
''' 
Variables:
    image_dir(str): directory of input data
    image_paths(list): list of all jpg files in the image_dir
    preprocess_path(str): diretory containing intermediaries
    processed_img_path (str): directory containing processed images
    processed_mask_path (str): directory containing generated masks
    processed_overlay_path (str): directory containing overlaid images
'''
mask_path = os.path.join('.', 'Big_Colonies_Masks')
overlay_path = os.path.join('.', 'Big_Colonies_Overlay')
segmented_mask_path = os.path.join('.', 'Mask4')

#### Toggle Additional Functions

In [24]:
def local_max(img, kernel):
    original_shape = img.shape[0]
    if img.shape[0] % kernel != 0:
        padding = kernel - (img.shape[0] % kernel)
        img = np.pad(img, ((0, padding), (0, padding)), mode='constant', constant_values=0)
    else:
        padding = 0
    out = np.zeros_like(img) # Get the output image
    for i in range(0, img.shape[0], kernel):
        for j in range(0, img.shape[1], kernel):
            if np.mean(img[i+kernel:i+2*kernel, j+kernel: j+2*kernel]) > np.mean(img[i:i+3*kernel, j:j+3*kernel]):
                out[i+kernel:i+2*kernel, j+kernel: j+2*kernel] = img[i+kernel:i+2*kernel, j+kernel:j+2*kernel]
    return out[:original_shape, :original_shape]
            

def solidity(img, thresh):
    '''
        Determining the solid and non-solid blobs and divide them into two masks
        
        Returns:
            mask_non_solid (numpy.ndarray): masks containing non solid blobs
            mask_solid (numpy.ndarray): masks containing solid blobs
    '''

    #####################################
    ## Split by solidity on sureFront1 ##
    #####################################
    '''
        Compute solidity (area/hull_area) and split the objects based on solidity.processed_overlay_path 

        Variables:
            contours(list): list of all object contours in the image.
            contours_solid(list): contours of objects that do not need further segmentation.
            contours_non_solid(list): contours of objects that need further segmentation due to bordering.

    '''
    contours, _ = cv.findContours(img, cv.RETR_TREE, cv.CHAIN_APPROX_SIMPLE) # detect all objects in sureFront1_inter
    contours_solid=[] # Does not need further segmentation
    contours_non_solid = [] # Need further segmentation
    for contour in contours:
        area = cv.contourArea(contour) # find area of a contour
        hull = cv.convexHull(contour) # find convex hull of a contour
        hull_area = cv.contourArea(hull) # find convex hull area
        if hull_area == 0:
            continue
        solidity = float(area)/hull_area
        if(solidity<thresh and area>2800): #threshold is 0.94
            contours_non_solid.append(contour)
        else:
            contours_solid.append(contour)

    ###########################################
    ## Further process the non-solid objects ##
    ###########################################
    '''
        Variables:
            mask_non_solid(numpy.ndarray): new mask for storing non-solid objects
            mask_solid(numpy.ndarray): new mask for storing solid objects
            sureFront(numpy.ndarray): final sure foreground with all but small objects

    '''
    contours_non_solid = tuple(contours_non_solid)
    contours_solid = tuple(contours_solid)
    mask_non_solid = np.zeros(img.shape, dtype=np.uint8)
    mask_solid = np.zeros(img.shape, dtype=np.uint8)
    cv.drawContours(mask_non_solid, contours_non_solid, contourIdx=-1, color=(255, 255, 255), thickness=cv.FILLED)
    cv.drawContours(mask_solid, contours_solid, contourIdx=-1, color=(255, 255, 255), thickness=cv.FILLED) #split the mask further
    
    return mask_non_solid, mask_solid


def distTrans(img, medianBlurKernel, localMaxKernel, openingKernel):
    # Distance Transform on the non-solid objects
    dist = cv.distanceTransform(img, distanceType=cv.DIST_L2, maskSize=3)
    dist = cv.normalize(dist, None, 255, 0, cv.NORM_MINMAX, cv.CV_8U)
    dist = cv.medianBlur(dist, medianBlurKernel)

    # Find local max from image
    kernel2 = cv.getStructuringElement(cv.MORPH_ELLIPSE, openingKernel)
    mask_non_solid = local_max(dist, localMaxKernel)
    _, mask_non_solid = cv.threshold(mask_non_solid, 1, 255, cv.THRESH_BINARY)
    mask_non_solid = cv.morphologyEx(mask_non_solid, cv.MORPH_OPEN, kernel2, iterations=3)
    # cv.imshow("img", cv.resize(mask_non_solid, (750, 750)))
    # cv.waitKey(0)
    
    return mask_non_solid


# def solidity_processing(img, thresh):
#     '''
#         For segmenting the hard-to-semgent non-solid objects:

#         1. Computes the solidity of each detected object
#         2. Uses the given threshold of "non-solid" objects and detect all "non-solid" objects
#         3. Further processes the "non-solid" objects

#     '''    
#     mask_non_solid, mask_solid = solidity(img, thresh)
#     while np.max(mask_non_solid) != 0:

#         mask_non_solid = distTrans(mask_non_solid, 35, 12, (15, 15))
        
#         mask_non_solid, mask_solid2 = solidity(mask_non_solid, thresh)

#         mask_solid = cv.bitwise_or(mask_solid, mask_solid2)
#         print(np.max(mask_non_solid))        
#         # mask_non_solid = distTrans(mask_non_solid, 35, 12, (9, 9))

#     sureFront = mask_solid #cv.bitwise_or(cv.bitwise_or(mask_non_solid, mask_solid), mask_solid)
    
#     return sureFront

def solidity_processing(img, thresh):
    '''
        For segmenting the hard-to-semgent non-solid objects:

        1. Computes the solidity of each detected object
        2. Uses the given threshold of "non-solid" objects and detect all "non-solid" objects
        3. Further processes the "non-solid" objects

    '''    
    mask_non_solid, mask_solid = solidity(img, thresh)

    mask_non_solid = distTrans(mask_non_solid, 35, 12, (15, 15))
    
    mask_non_solid2, mask_solid2 = solidity(mask_non_solid, thresh)
    
    mask_non_solid2 = distTrans(mask_non_solid2, 35, 12, (9, 9))

    sureFront = cv.bitwise_or(cv.bitwise_or(mask_non_solid2, mask_solid2), mask_solid)
    
    return sureFront


# maybe repeat this twice

#### Part I - Colonies Segmentation
Colonies are segmented with an augmented divide-conquer watershed algorithm.\
\
Main techniques used:
1. connectedComponent: separating small colonies from large colonies
1. Marker-Based Watershed: segment colonies by watershed, markers generated from sure foreground and background.

In [ ]:
%%skip_if segment_colonies == False


# Directories as sorted list
mask_dir = sorted(os.listdir(mask_path), key=numericalSort)

# Kernel used for morphological transformation
kernel = np.ones((3,3), np.uint8)

# Contours of segmentation
contours_list = {}

for filename_mask in mask_dir:
    print('Now Segmenting:', filename_mask)
    filename_overlay = "Overlay" + filename_mask[4:]
    mask = cv.cvtColor(cv.imread(os.path.join(mask_path, filename_mask)).astype(np.uint8), cv.COLOR_RGB2GRAY) # converting mask to uint8 and grayscale
    overlay = cv.imread(os.path.join(overlay_path, filename_overlay))

    ###################################################################
    ## Inverse grayscale of the overlay to threshold the colony core ##
    ###################################################################
    '''
        Variables:
            overlay_inverse(numpy.ndarray): intermediate threshold image for finding sure foreground and background, the inverse grayscale image of overlay and mask
            sureBack(numpy.ndarray): sure background, obtained from overlay_inverse
            sureFront1_inter(numpy.ndarray): intermediate sure background, obtained from overlay_inverse
            sureFront2_inter(numpy.ndarray): intermediate sure background for keeping small colonies, obtained from overlay_inverse

    '''
    overlay_org = cv.medianBlur(cv.cvtColor(overlay.copy(), cv.COLOR_BGR2GRAY), 15)
    overlay_inverse = cv.bitwise_not(overlay_org)
    overlay_inverse = cv.medianBlur(overlay_inverse, 5)
    overlay_inverse = cv.bitwise_and(overlay_inverse, mask)
    # cv.imshow("img", cv.resize(overlay_inverse, (750, 750)))
    # cv.waitKey(0)


    # Binary thresholding    
    if core == True:
        sureFront1_inter = cv.threshold(overlay_inverse, 239, 255, cv.THRESH_BINARY)[1] #240 # For obtaining sure foreground
    else:
        sureFront1_inter = cv.threshold(overlay_inverse, 7, 255, cv.THRESH_BINARY)[1]
    sureFront2_inter = cv.threshold(overlay_inverse, 7, 255, cv.THRESH_BINARY)[1] #7 # For obtaining small colonies of the sure foreground
    sureBack = cv.dilate(cv.threshold(overlay_inverse, 115, 255, cv.THRESH_BINARY)[1], kernel, iterations=5) #110 # Sure background


    ################################################
    ## Morphological Transformation on sureFront2 ##
    ################################################
    '''
        Variables:
            sureFront2_inter(numpy.ndarray): intermediate sure background for keeping small colonies
            sureFront2(numpy.ndarray): sure background containing only small colonies less than 7000 pixels big

    '''
    sureFront2_inter = cv.erode(sureFront2_inter, kernel, iterations=8) # use RGB
    sureFront2_inter = cv.dilate(sureFront2_inter, kernel, iterations=2) # use RGB
    

    # Remove big objects from the mask
    nb_blobs, mask_separated_blobs, stats, _ = cv.connectedComponentsWithStats(sureFront2_inter)
    sizes = stats[:, cv.CC_STAT_AREA]
    sureFront2 = np.zeros_like(mask_separated_blobs)
    for i in range(1, nb_blobs):
        if sizes[i] <= 7000 and sizes[i] >= 200: #exclude big objects
            sureFront2[mask_separated_blobs == i] = 255
    sureFront2 = img_as_ubyte(sureFront2)
    

    ################################################
    ## Morphological Transformation on sureFront1 ##
    ################################################
    '''
        Variables:
            sureFront1_inter(numpy.ndarray): intermediate sure background for keeping small colonies
            sureFront2(numpy.ndarray): sure background containing only small colonies less than 7000 pixels big

    '''
    if core == True:
        temp_kernel = cv.getStructuringElement(cv.MORPH_CROSS, (12, 12))
        sureFront1_inter = cv.morphologyEx(sureFront1_inter, cv.MORPH_OPEN, temp_kernel)
        sureFront1 = img_as_ubyte(morphology.remove_small_holes(sureFront1_inter, area_threshold=2000))
    else:
        sureFront1 = solidity_processing(sureFront1_inter, 0.94)


    ############################
    ## Marker-based Watershed ##
    ############################
    '''
        Variables:
            sureFront(numpy.ndarray): final sure foreground.
            unsure(numpy.ndarray): unsure regions to differentiate foreground and background.
            markers(numpy.ndarray): markers specifying the regions to flood for watershed.
            markers_ws(numpy.ndarray): markers outputed by marker-based watershed algorithm. Boundary is set to -1

    '''
    sureFront = cv.bitwise_or(sureFront1, sureFront2) #merging sureFront1 and sureFront2
    unsure = cv.subtract(sureBack, sureFront)

    # Marker labelling
    _, markers = cv.connectedComponents(sureFront) # Labels background of the image with 0, and other objects are labelled with natural numbers
    markers += 1 # Give different label to the sure background
    markers[unsure==255] = 0 # Make sure the unsure ground is labelled 0, can be detected with watershed
    # cv.imshow("img", cv.resize(unsure, (750, 750)))
    # cv.waitKey(0)

    # Watershed
    mask = cv.imread(os.path.join(mask_path, filename_mask)) # read the original mask (not grayscaled)
    markers_ws = cv.watershed(mask, markers) # takes in the image and the markers, where the unknown region is labelled as 0


    ##################
    ## Segmentation ##
    ##################
    '''
        Variables:
            markers_ws(numpy.ndarray): markers outputed by marker-based watershed algorithm.
            unsure(numpy.ndarray): unsure regions to differentiate foreground and background.
            markers(numpy.ndarray): markers specifying the regions to flood for watershed.

    '''
    markers_ws = markers_ws.astype(np.uint8)
    _, thresh = cv.threshold(markers_ws, 0, 255, cv.THRESH_BINARY|cv.THRESH_OTSU) # Detect edges that includes the newly segmented lines
    contours, _ = cv.findContours(thresh, cv.RETR_LIST, cv.CHAIN_APPROX_SIMPLE) # Find the detected edges

    # etching the segmentation onto the mask, converting the image into grayscale
    cv.cvtColor(cv.drawContours(mask, contours, -1, (0, 0, 0), 5), cv.COLOR_RGB2GRAY) 
    

    ############################
    ## Saving Segmented Masks ##
    ############################
    '''
        mask_name(String): the name of a segmented mask
                           the format of the naming is (S stands for segmented):
                                                        SMask_Sample_category-number.jpg          
    '''
    mask_name = "S" + filename_mask
    cv.imwrite(os.path.join(segmented_mask_path, mask_name), mask)

Now Segmenting: Mask_Sample_1-1.jpg
Now Segmenting: Mask_Sample_1-2.jpg
Now Segmenting: Mask_Sample_1-3.jpg
Now Segmenting: Mask_Sample_1-4.jpg
Now Segmenting: Mask_Sample_1-5.jpg
Now Segmenting: Mask_Sample_1-6.jpg
Now Segmenting: Mask_Sample_1-7.jpg
Now Segmenting: Mask_Sample_1-8.jpg
Now Segmenting: Mask_Sample_1-9.jpg
Now Segmenting: Mask_Sample_1-10.jpg
Now Segmenting: Mask_Sample_1-11.jpg
Now Segmenting: Mask_Sample_1-12.jpg
Now Segmenting: Mask_Sample_2-10.jpg
Now Segmenting: Mask_Sample_2-11.jpg
Now Segmenting: Mask_Sample_2-12.jpg
Now Segmenting: Mask_Sample_2-13.jpg
Now Segmenting: Mask_Sample_2-14.jpg
Now Segmenting: Mask_Sample_2-15.jpg
Now Segmenting: Mask_Sample_3-1.jpg
Now Segmenting: Mask_Sample_3-2.jpg
Now Segmenting: Mask_Sample_3-3.jpg
Now Segmenting: Mask_Sample_3-4.jpg
Now Segmenting: Mask_Sample_3-5.jpg
Now Segmenting: Mask_Sample_3-6.jpg
Now Segmenting: Mask_Sample_3-7.jpg
Now Segmenting: Mask_Sample_3-8.jpg
Now Segmenting: Mask_Sample_3-9.jpg
Now Segmenting: Mas

: 

#### Part II - Colony Counting
Colonies are counted from the segmented mask with moments.\
\
Main techniques used:
1. connectedComponent: separating small colonies from large colonies
1. ConvexHull: segment colonies by convex hull

In [6]:
ground_truth = [[113, 90, 99, 87, 96, 93, 92, 95, 96, 76, 60, 63, 10, 11, 10, 1, 1, 1],
                [145, 175, 160, 167, 156, 120, 395, 446, 436, 107, 101, 92, 43, 28, 45, 4, 3, 6],
                [103, 112, 72, 74, 76, 100, 62, 74, 79, 71, 83, 79, 8, 11, 10, 3, 2, 12],
                [170, 176, 160, 139, 137, 156, 101, 98, 93, 96, 102, 87, 27, 29, 20, 7, 4, 4],
                [165, 159, 129, 149, 164, 160, 90, 89, 103, 116, 127, 128, 18, 18, 16, 3, 8, 9],
                [139, 140, 123, 117, 106, 121, 89, 100, 79, 91, 100, 102, 14, 22, 9, 7, 10, 7],
                [160, 153, 144, 140, 140, 151, 149, 143, 146, 166, 168, 169, 25, 20, 20, 14, 11, 9],
                [157, 160, 117, 138, 140, 155, 144, 118, 117, 138, 125, 108, 27, 29, 25, 6, 11, 10],
                [169, 156, 173, 217, 171, 170, 123, 125, 109, 166, 143, 173, 32, 19, 21, 8, 6, 4],
                [153, 156, 135, 157, 138, 163, 94, 92, 72, 119, 117, 92, 31, 31, 21, 4, 9, 8],
                [147, 158, 143, 156, 149, 119, 118, 143, 121, 168, 175, 144, 28, 27, 18, 10, 8, 13],
                [137, 115, 115, 126, 122, 136, 93, 105, 114, 137, 119, 142, 28, 29, 25, 8, 12, 7],
                [228, 240, 294, 259, 243, 239, 1, 1, 0],
                [114, 98, 115, 104, 98, 121, 99, 75, 75, 59, 76, 62, 9, 9, 14, 4, 1, 0],
                [220, 202, 213, 176, 209, 201, 112, 127, 101, 65, 62, 64, 18, 23, 31, 5, 3, 5],
                [145, 152, 167, 164, 167, 166, 114, 82, 74, 59, 12, 10, 13, 1, 3, 4],
                [61, 59, 62, 60, 67, 57, 30, 19, 36, 33, 45, 32, 6, 7, 16, 16, 16, 7, 15, 15, 17, 33, 28, 28],
                [103, 95, 87, 87, 83, 90, 44, 48, 51, 45, 63, 55, 13, 19, 17, 19, 20, 21, 6, 7, 2, 4, 2, 4],
                [74, 72, 93, 65, 76, 87, 60, 55, 49, 59, 49, 66, 16, 11, 9, 15, 12, 14, 4, 2, 5, 7, 4, 8],
                [8, 11, 13, 6, 10, 5, 46, 55, 53, 69, 67, 52, 8, 11, 14, 6, 19, 12, 12, 18, 11, 14, 15, 11],
                [70, 90, 80, 96, 93, 76, 68, 63, 65, 78, 75, 59, 10, 6, 10, 8, 16, 9, 3, 2, 4, 6, 5, 4],
                [43, 48, 46, 34, 41, 41, 12, 16, 8, 25, 24, 20, 3, 7, 12, 6, 7, 5, 4, 2, 2, 1, 2, 0],
                [133, 130, 99, 105, 117, 110, 34, 49, 55, 48, 55, 65, 17, 22, 14, 14, 16, 13, 2, 2, 3, 8, 1, 6],
                [29, 40, 5, 12, 9, 6],
                [118, 102, 108, 101, 112, 113, 46, 45, 35, 47, 54, 45, 11, 16, 16, 12, 20, 17, 3, 2, 1, 4, 3, 1],
                [144, 117, 147, 149, 125, 122, 56, 73, 52, 40, 55, 51, 21, 29, 30, 23, 19, 32, 9, 5, 3, 8, 3, 5],
                [131, 148, 160, 155, 158, 132, 96, 110, 79, 116, 123, 108, 46, 33, 46, 40, 45, 42, 42, 26, 40, 42, 30, 36], #27
                [157, 144, 143, 142, 159, 163, 78, 78, 70, 149, 153, 124, 53, 43, 44, 49, 45, 45, 28, 26, 22, 24, 20, 20],
                [134, 128, 108, 121, 111, 105, 94, 85, 84, 110, 126, 116, 53, 48, 49, 51, 51, 40, 41, 46, 36, 38, 42, 37],
                [159, 134, 163, 165, 179, 147, 94, 95, 92, 158, 149, 158, 41, 39, 39, 38, 45, 47, 36, 30, 32, 34, 36, 25],
                [181, 136, 161, 158, 155, 130, 70, 74, 72, 128, 140, 125, 51, 42, 46, 52, 57, 48, 18, 25, 27, 20, 23, 19],
                [137, 141, 137, 147, 158, 124, 82, 75, 80, 166, 143, 130, 28, 34, 39, 47, 39, 46, 17, 25, 21, 19, 22, 20],
                [146, 146, 164, 125, 150, 160, 111, 110, 98, 134, 120, 129, 64, 46, 52, 60, 43, 55, 36, 34, 42, 31, 28, 36],
                [109, 107, 110, 78, 139, 120, 40, 57, 73, 60, 65, 66, 14, 29, 31, 26, 22, 25, 8, 8, 10, 11, 16, 7],
                [131, 114, 139, 110, 125, 123, 104, 105, 118, 83, 95, 90, 13, 19, 18, 15, 17, 12, 10, 12, 7, 7, 7, 7],
                [114, 164, 154, 141, 129, 118, 112, 122, 119, 160, 184, 175, 30, 36, 34, 17, 33, 33, 19, 17, 8, 8, 17, 18],
                [119, 130, 132, 135, 124, 140, 87, 107, 109, 132, 134, 100, 22, 14, 22, 19, 23, 17, 10, 4, 8, 9, 6, 4],
                [146, 143, 127, 148, 123, 112, 81, 102, 82, 90, 106, 101, 35, 35, 28, 31, 34, 28, 9, 9, 8, 9, 11, 9]]

In [7]:
%%skip_if count_colonies == False


def accuracy_calc(ground_truth, experimental):
    '''
        Calculate the accuracy of the segmented mask from the manually counted colony count.

        Variables:
            ground_truth(list): a list of manually counted colony count for accuracy calculations.
            experimental(list): a list of counts from algorithm
            batch(int): for indexing the batch in ground_truth and experimental lists.

        Return:
            accuracy(double): average accuracy calculated with the masks in a batch
            
    '''
    if ground_truth == 0:
        percent_diff = 0 # discount the masks with no colonies
    else:
        percent_diff = abs((experimental - ground_truth)) / ground_truth
        
    if(percent_diff>1): 
        percent_diff = 1
    accuracy = (1 - percent_diff) * 100
    return accuracy


####################
## Count Colonies ##
####################
''' 
    Variables:
        experimental(list): a list of counts from algorithm
        total_accuracy(list): a list of accuracies of each batch
        batch(int): for indexing the batch in ground_truth and experimental lists.
        sample(int): for indexing a particular sample in a given batch.
        mask(numpy.ndarray): segmented mask
        counter(int): for counting the number of colonies in a mask

'''
experimental = [] # storing all of the experimental counts
total_accuracy = []
mask_dir = sorted(os.listdir(segmented_mask_path), key=numericalSort)

for filename in mask_dir:
    mask_org = cv.imread(os.path.join(segmented_mask_path, filename))
    mask = cv.cvtColor(mask_org, cv.COLOR_BGR2GRAY).astype(np.uint8)

    ## Find contours of the segmented colonies ##
    _, thresh = cv.threshold(mask, 15, 255, cv.THRESH_BINARY|cv.THRESH_OTSU)
    contours, _ = cv.findContours(thresh, cv.RETR_LIST, cv.CHAIN_APPROX_SIMPLE)
    cv.drawContours(mask_org, contours, -1, (0, 255, 0), 5)
    destination = os.path.join('.', 'Data', 'HCT116_Dataset', 'contour_small', filename)
    cv.imwrite(destination, mask_org)


    ## Count the colonies based on the contours ##
    counter = 0
    for contour in contours:
        # Compute moments of the contour
        M = cv.moments(contour)
        if M['m00'] != 0:
            counter += 1 # Detected a colony

    print(f"{filename} has {counter} colonies")
    
    
    ## Calculate the accuracy of a batch ##
    if test_accuracy == True:
        batch = int(filename[13]) - 1
        try:
            sample = int(filename[15:17]) - 1
        except:
            sample = int(filename[15:16]) - 1
        accuracy = accuracy_calc(ground_truth=ground_truth[batch][sample], experimental=counter)
        print(accuracy)
        total_accuracy.append(accuracy)


## Calculate accuracy of the entire dataset ##
if test_accuracy == True:
    avg_accuracy = sum(total_accuracy) / len(total_accuracy)
    print(f"The averaged accuracy for the entire dataset is {avg_accuracy}")


SMask_Sample_1-1.jpg has 84 colonies
74.33628318584071
SMask_Sample_1-2.jpg has 76 colonies
84.44444444444444
SMask_Sample_1-3.jpg has 62 colonies
62.62626262626263
SMask_Sample_1-4.jpg has 73 colonies
83.9080459770115
SMask_Sample_1-5.jpg has 77 colonies
80.20833333333334
SMask_Sample_1-6.jpg has 73 colonies
78.49462365591397
SMask_Sample_1-7.jpg has 76 colonies
82.6086956521739
SMask_Sample_1-8.jpg has 81 colonies
85.26315789473684
SMask_Sample_1-9.jpg has 83 colonies
86.45833333333334
SMask_Sample_1-10.jpg has 66 colonies
86.8421052631579
SMask_Sample_1-11.jpg has 53 colonies
88.33333333333333
SMask_Sample_1-12.jpg has 47 colonies
74.60317460317461
SMask_Sample_2-1.jpg has 100 colonies
68.96551724137932
SMask_Sample_2-2.jpg has 120 colonies
68.57142857142857
SMask_Sample_2-3.jpg has 103 colonies
64.375
SMask_Sample_2-4.jpg has 114 colonies
68.26347305389223
SMask_Sample_2-5.jpg has 99 colonies
63.46153846153846
SMask_Sample_2-6.jpg has 97 colonies
80.83333333333333
SMask_Sample_2-7.

#### Miscellaneous Codes

In [17]:
# ## Split by solidity on sureFront1 ##
# '''
#     Compute solidity (area/hull_area) and split the objects based on solidity.

#     Variables:
#         contours(list): list of all object contours in the image.
#         contours_solid(list): contours of objects that do not need further segmentation.
#         contours_non_solid(list): contours of objects that need further segmentation due to bordering.

# '''
# contours, _ = cv.findContours(sureFront1_inter, cv.RETR_TREE, cv.CHAIN_APPROX_SIMPLE) # detect all objects in sureFront1_inter
# contours_solid=[] # Does not need further segmentation
# contours_non_solid = [] # Need further segmentation
# for contour in contours:
#     area = cv.contourArea(contour) # find area of a contour
#     hull = cv.convexHull(contour) # find convex hull of a contour
#     hull_area = cv.contourArea(hull) # find convex hull area
#     solidity = float(area)/hull_area
#     if(solidity<0.94 and area>500): #threshold is 0.94
#         contours_non_solid.append(contour)
#     else:
#         contours_solid.append(contour)

# contours_non_solid = tuple(contours_non_solid)
# contours_solid = tuple(contours_solid)


# ## Further process the non-solid objects ##
# '''
#     Variables:
#         mask_non_solid(numpy.ndarray): new mask for storing non-solid objects
#         mask_solid(numpy.ndarray): new mask for storing solid objects
#         kernel2(numpy.ndarray): kernel specifically for smoothing and segmenting objects
#         sureFront1(numpy.ndarray): final sure foreground with all but small objects

# '''
# mask_non_solid = cv.cvtColor(np.zeros(overlay.shape, dtype=np.uint8), cv.COLOR_RGB2GRAY)
# mask_solid = cv.cvtColor(np.zeros(overlay.shape, dtype=np.uint8), cv.COLOR_RGB2GRAY)
# cv.drawContours(mask_non_solid, contours_non_solid, contourIdx=-1, color=(255, 255, 255), thickness=cv.FILLED)
# cv.drawContours(mask_solid, contours_solid, contourIdx=-1, color=(255, 255, 255), thickness=cv.FILLED) #split the mask further

# # Distance Transform on the non-solid objects
# dist_transform = cv.distanceTransform(mask_non_solid,cv.DIST_L2,5)
# _, mask_non_solid = cv.threshold(dist_transform,0.15*dist_transform.max(),255,0)
# mask_non_solid = cv.normalize(mask_non_solid, None, 255, 0, cv.NORM_MINMAX, cv.CV_8U)
# kernel2 = cv.getStructuringElement(cv.MORPH_ELLIPSE,(5,5))
# # kernel1 = cv.getStructuringElement(cv.MORPH_CROSS,(7,7))
# # new_mask = cv.dilate(new_mask, kernel2, iterations=2)
# mask_non_solid = cv.erode(mask_non_solid, kernel2, iterations=9)
# mask_non_solid = cv.morphologyEx(mask_non_solid, cv.MORPH_CLOSE, kernel, iterations=3)
# # mask_non_solid = img_as_ubyte(mask_non_solid) # convert image to uint8 format
# # mask_solid = cv.dilate(mask_solid, kernel2, iterations=2)
# sureFront1 = cv.bitwise_or(mask_non_solid, mask_solid)




# ## Eccentricity ##
# for contour in contours:
#         # Compute moments of the contour
#         M = cv.moments(contour)
#         print(M)
#         print(awdwdsd)
#         if M['m00'] != 0:
#             eccentricity = (M['m20'] + M['m02'] + math.sqrt(( M['m20'] - M['m02'])**2 + 4 * (M['m11']**2))) / (M['m20'] + M['m02'] - math.sqrt(( M['m20'] - M['m02'])**2 + 4 * (M['m11']**2))) # https://users.cs.cf.ac.uk/Dave.Marshall/Vision_lecture/node36.html
#             cx = int(M['m10'] / M['m00'])
#             cy = int(M['m01'] / M['m00'])
#             # Draw circle at center of mass
#             cv.circle(mask, (cx, cy), 5, (0, 255, 0), -1)
#         print(eccentricity)
